# Week 11: More Linear Regression

**Tomo Parins-Fukuchi**

## Exam Info

<div style="float:left; width:45%;">
<ul>
<li>Final Exam is on Thursday, April 15, 2026 from 9 am ET to 12 pm ET.</li>
<li>The final exam will be written in a supervised, in-person setting, in computer labs in Bahen Centre.</li>
<li>Bring your Tcard or some other form of goernment-issued photo ID. The <a href="https://artsci.calendar.utoronto.ca/term-work-tests-and-final-exams#final-exams" target="_blank">FAS rules on in-person final exams</a> will apply.</li>
<li>Each student will use a lab computer, <em>not their own laptop,</em> to complete the exam.</li>
<li>You'll login with your UTORid and password, and will also need your U of T email address to login to JupyterHub.</li>
</ul>
</div>
<div style="float:left; width:45%;">
<ul>
<li><strong>Allowed aids</strong></li>
    <ul>
    <li>The exam will be written on a Jupyter notebook.</li>
    <li>You'll have access to the EEB125 course website, JupyterHub (including all of your course work from the semester), and MarkUs. However, you won't have access to any other websites (e.g., Quercus, Google, Facebook), and of course will not be allowed to communicate with anyone else during the exam.</li>
    <li>You may bring additional course notes on paper (handwritten or printed).</li>
    </ul>
</ul>
</div>

## Data Science Methods

**Hypothesis testing** covered how to:

+ test if a percentage (statistic) is different from some hypothesized values.
+ test if two different groups have different means (or median).

**Confidence intervals** covered how to:
+ get a range of values which estimate a parameter (i.e., mean, median, percentage).
+ get a range of values which estimate the difference in means (or medians) of two different groups.

**Linear regression** we covered how to:
+ look at the relationship between two columns in a data frame. (e.g., the relationship between height and mass of mammals).

## Linear Regression

### Basic idea

- Linear regression is a useful technique for creating models to explain relationships between variables. 

- The dependent variable must be numeric, and have meaningful numeric values. 

- The independent variables can be interval or categorical variables.

We have two variables and want to estimate a line to describe their relationship:

<img src="line.png" width="400" style="float:left; margin: 5px 25px;">

We call the estimated line $\hat{Y} = m X + b$, where:

+ $m$ is the estimated slope
+ $b$ is the estimated y-intercept


<img src="lines.png" width="400" style="float:left; margin: 5px 25px;">

We are testing if the slope is non-zero, where a linear relationship exists between the independent and dependent variables

**Recall, the summary table will output a p-value for testing: $H_0: slope = 0$.**


### Linear regression

+ P-value is small: there is strong evidence the slope is not 0 
    - a linear relationship may exist
+ P-value is large: there is weak evidence the slope is not 0 
    - poor evidence for a linear relationship

### "Linear with noise" - from last week

- Last week we did a linear regression on data that was almost perfectly linear
- Let's build on this example to assess whether the model is appropriate

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(717) 

data = {"depvar" : np.arange(start=0, stop=8, step=1) + 2 + np.random.uniform(low=0, high=2, size=8),
        "indvar" : np.arange(start=0, stop=8, step=1)}

df = pd.DataFrame(data)

df

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(x=df["indvar"], y=df["depvar"])
plt.xlabel("indvar")
plt.ylabel("depvar");

In [ ]:
import statsmodels.formula.api as smf

regmod = smf.ols("depvar ~ indvar", data=df) 

regmod_fit = regmod.fit() 


### Linear regression

So, now the relationship isn't perfectly linear, but close.  The equation of this regression line is:

$$Y = 3.021243 X + 1.066628 $$

In [ ]:
plt.scatter(x=df["indvar"], y=df["depvar"])
plt.axline(xy1=(0, regmod_fit.params.iloc[0]), slope=regmod_fit.params.iloc[1])
plt.xlabel("indvar")
plt.ylabel("depvar")
plt.show()

### Fitted/predicted values

- We may wish to know what values the model predicts for the dependent variable given a value for the independent variable
- We can find these **fitted** or **predicted** values by just plugging our X values into the linear equation with our parameters
 
$$Y = 1.0666X + 3.0212 $$

### Fitted values


The fitted value for the first row of `df` is:

In [ ]:
regmod_fit.params.iloc[0] + regmod_fit.params.iloc[1] * df.iloc[0, 1] # 1.1127 + 0.9644 * 0.2710


### Predicted values

- If the linear regression model is used on an independent variable that is not in the data set used to build the model then it's often referred to as a **predicted value**

In [ ]:
regmod_fit.params.iloc[0] + regmod_fit.params.iloc[1] * 9 # e.g., independent variable at 9

### Fitted/predicted values

To extract the fitted values from a regression model use the `fittedvalues` function in `statsmodels`.

In [ ]:
regmod_fit.fittedvalues

### Fitted/predicted values

- The **residual** is how far above or below the line a point is 
    - i.e., it is the dependent variable minus the fitted value

- So, for the first row the residual is:

In [ ]:
2 - 1.37398

### Residuals

To extract the residuals from a regression model use the `resid` attribute of the fitted object (i.e., `regmod_fit`).

In [ ]:
regmod_fit.resid

## Model evaluation

- The residuals offer useful information if we want to see how well our regression model explains our data
 
- **R-squared** provides one measure of the model's fit to our data
     + R-squared of 0 indicates a poor fit
     + R-squared of 1 indicates a perfect fit

In [ ]:
plt.scatter(x=df["indvar"], y=df["depvar"])
plt.axline(xy1=(0, regmod_fit.params.iloc[0]), slope=regmod_fit.params.iloc[1])
plt.xlabel("indvar")
plt.ylabel("depvar")
plt.text(0.05, 0.9, f"R^2: {regmod_fit.rsquared:.3f}", transform=plt.gca().transAxes)
plt.show()

### R-squared

- $R^2$ is computed using the ratio of two terms:
    - **Total sum of squares**: sum of squared differences between each `depvar` ($y_i$) and the mean `depvar` ($\bar{y}$)
    $$ TSS = \sum{(y_i - \bar{y})^2} $$
        - How much noise exists in the dependent variable?
    - **Residual sum of squares**: total sum of all squared residuals
- _How much of the variation in the dependent variable is explained by the independent variable?_

### R-squared

- $R^2$ is a very useful measure
- Like all measures, it can be broken

In [ ]:
np.random.seed(717) # set the seed so that it's reproducible

data = {"depvar" : (0.001 * np.arange(start=0, stop=8, step=1)) + 2.0 + np.random.normal(0, 0.001, size=8),
        "indvar" : np.arange(start=0, stop=8, step=1)}

df = pd.DataFrame(data)

plt.scatter(x=df["indvar"], y=df["depvar"])
plt.ylim(1, 3.5)
plt.xlabel("indvar")
plt.ylabel("depvar");

In [ ]:
regmod = smf.ols("depvar ~ indvar", data=df) # setup the model

regmod_fit = regmod.fit() # estimate/fit the model 

In [ ]:
plt.scatter(x=df["indvar"], y=df["depvar"])
plt.axline(xy1=(0, regmod_fit.params.iloc[0]), slope=regmod_fit.params.iloc[1])
plt.xlabel("indvar")
plt.ylabel("depvar")
plt.ylim(1, 3.5)
plt.text(0.05, 0.9, f"R^2: {regmod_fit.rsquared:.3f}", transform=plt.gca().transAxes)
plt.show()

### What is happening here?

- There is very little noise around the points
- What variation does exist in `y` is well explained by `x`, even though the line is basically flat
- This changes if there is more significant noise around `y`

In [ ]:
np.random.seed(717) 

data = {"depvar" : (0.001 * np.arange(start=0, stop=8, step=1)) + 2.0 + np.random.normal(0, 0.05, size=8),
        "indvar" : np.arange(start=0, stop=8, step=1)}

df = pd.DataFrame(data)

plt.scatter(x=df["indvar"], y=df["depvar"])
plt.ylim(1, 3.5)
plt.xlabel("indvar")
plt.ylabel("depvar");

In [ ]:
regmod = smf.ols("depvar ~ indvar", data=df)
regmod_fit = regmod.fit()  

In [ ]:
plt.scatter(x=df["indvar"], y=df["depvar"])
plt.axline(xy1=(0, regmod_fit.params.iloc[0]), slope=regmod_fit.params.iloc[1])
plt.xlabel("indvar")
plt.ylabel("depvar")
plt.ylim(1, 3.5)
plt.text(0.05, 0.9, f"R^2: {regmod_fit.rsquared:.3f}", transform=plt.gca().transAxes)
plt.show()


<img src="lines.png" width="400" style="float:center; margin: 5px 25px;">



### Evaluating linear regression

- The fit of a linear regression model depends on two factors:
    - The slope of the line (+ or -) 
        - in what direction and how linearly does `x` affect `y`?
    - The "spread" of the residuals around the line
        - How much "extra" noise exists around the relationship predicted by the model? 

### Fitted values vs residuals

One other way to evaluate a regression model is to plot the residuals against the fitted values


In [ ]:
## Re-simulate our data for our "nearly linear" example

np.random.seed(717) 

data = {"depvar" : np.arange(start=0, stop=8, step=1) + 2 + np.random.uniform(low=0, high=2, size=8),
        "indvar" : np.arange(start=0, stop=8, step=1)}

df = pd.DataFrame(data)

regmod = smf.ols("depvar ~ indvar", data=df) 

regmod_fit = regmod.fit() 

In [ ]:
plt.scatter(regmod_fit.fittedvalues, regmod_fit.resid)
plt.axhline(y=0, color="red", linestyle="dotted")
plt.xlabel("fitted values")
plt.ylabel("residuals")
plt.show()

### Fitted values vs residuals

- This is essentially a plot of the **signal** captured by our model versus the remaining **noise** in the data
- Under linear regression, our error term ($\epsilon$) assumes that the noise is distributed evenly
- Lower remaining error (values closer to zero) is also suggestive of a stronger fit 

### Fitted values vs residuals

 - We want a plot of the residuals to look like the first graphic below:

<img src="residuals.png" width="400">

## Exploring regression models using the mammal data

- Pantheria data contains data on longevity and body mass.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np

pantheria = pd.read_csv("pantheria.txt", sep="\t")
pantheria.head()

In [ ]:
cols = ["5-1_AdultBodyMass_g", "17-1_MaxLongevity_m", "9-1_GestationLen_d", "MSW05_Binomial", "MSW05_Order"]

panthdat = pantheria[cols]

colnames = {cols[0] : "bodymass",
            cols[1] : "longevity",
            cols[2] : "gestation",
            cols[3] : "name",
            cols[4] : "order"}

panthdata = panthdat.copy()

panthdata.rename(columns=colnames, inplace=True)

panthdata.head()

Let's look at the distribution of `"bodymass"`, `"longevity"`, and `"gestation"`

In [ ]:
panthdata.hist(column=["bodymass", "longevity", "gestation"], 
               bins=15, color="grey", edgecolor="white", grid=False);

### Skewness

- These are all skewed
- But body mass in particular is pretty extreme
- It looks pretty wonky in a scatter plot

In [ ]:
plt.scatter(y=panthdata["gestation"], x=panthdata["bodymass"], alpha=.2)
plt.xlabel("Body mass (g)")
plt.ylabel("Gestation (days)");

## Regression model 1: Body mass and gestation

- We can fit a regression model to this-- but will it make any sense?
- Let's investigate.


In [ ]:
plt.scatter(y=panthdata["gestation"], x=panthdata["bodymass"], alpha=.2)
plt.xlabel("Body mass (g)")
plt.ylabel("Gestation (days)");

In [ ]:
reg_mod3 = smf.ols("gestation~bodymass", data=panthdata) # setup the model
reg_mod3_fit = reg_mod3.fit() # estimate/fit the model 
reg_mod3_summ = reg_mod3_fit.summary()
reg_mod3_summ.tables[1]

- The regression equation is: $\texttt{gestation} = 128 + 0.00000308 \times \texttt{body.mass}$
- The p-value for the slope indicates that body mass has a non-zero slope when predicting gestation.

In [ ]:
import seaborn as sns

sns.regplot(x="bodymass", y="gestation", data=panthdata, ci=None, scatter_kws={"alpha":0.1}) 
plt.text(0.7, 0.2, f"R^2: {reg_mod3_fit.rsquared:.3f}", transform=plt.gca().transAxes)
plt.show()

### Fitted values vs residuals

In [ ]:
plt.scatter(x=reg_mod3_fit.fittedvalues , y=reg_mod3_fit.resid, alpha=.3)
plt.axhline(y=0, color="red", linestyle="dotted")
plt.xlabel("Fitted values")
plt.ylabel("Residuals");

## Regression model 2: Length of longevity on gestation

- Let's explore another example where the skew is less extreme

In [ ]:
panthdata.hist(column=["longevity", "gestation"], 
               bins=15, color="grey", edgecolor="white", grid=False);

In [ ]:
import statsmodels.formula.api as smf

reg_mod1 = smf.ols("gestation ~ longevity", data=panthdata) # setup the model

reg_mod1_fit = reg_mod1.fit() 

### Statistical summary of the regression model

In [ ]:
reg_mod1_summ = reg_mod1_fit.summary()
reg_mod1_summ.tables[1]

In [ ]:
sns.regplot(x="longevity", y="gestation", data=panthdata, ci=None, scatter_kws={"alpha":0.1}) 
plt.xlabel("Longevity (months)")
plt.ylabel("Gestation (days)");

### Longevity vs gestation time

- The regression equation is: $\texttt{gestation} = 39.04 + 0.43\times \texttt{longevity}$
- The **slope** indicates that for a 1 month **increase** in longevity, we expect the gestational period to increase by 0.43 days.
- The **y-intercept** indicates that when longevity is 0 months, gestational period is 39 days.
- The **p-value** for the slope is 0 indicating the slope **is** significantly different from 0

### Assessing model fit


In [ ]:
reg_mod1_fit.rsquared

### Assessing model fit

Do you notice any patterns?

In [ ]:
plt.scatter(x=reg_mod1_fit.fittedvalues , y=reg_mod1_fit.resid, alpha=.3)
plt.axhline(y=0, color="red", linestyle="dotted")
plt.xlabel("Fitted values")
plt.ylabel("Residuals");

### Data Transformation

- Many statistical models assume that data are basically symmetrical
- It is very common to **transform** data that is distributed unevenly
- One way to deal with right-skewed data is to transform the x-values using $\log_{e}(x)$ to stretch out the scale on the left


In [ ]:
x = np.power(10, np.arange(start=1, stop=10, step=1))
logx = np.log(x)
print(pd.DataFrame(logx, x))

### Log transformation

We can compute $log(x)$ of each variable using `np.log()`

In [ ]:
np.log(panthdata[["bodymass", "longevity", "gestation"]]).head()

In [ ]:
panthdata[["bodymass_log", "longevity_log", "gestation_log"]] = np.log(panthdata[["bodymass", "longevity","gestation"]])
panthdata.hist(column=["bodymass_log", "longevity_log", "gestation_log"], 
               bins=15, color="grey", edgecolor="white", grid=False);

### Missing values

It will also be helpful to drop missing values.  This can be done using the `pandas` function `dropna` with the parameter `inplace=True`, so that it modifies the existing `DataFrame`.

In [ ]:
print(panthdata.isna().sum()) # check for missing values

In [ ]:
panthdata.dropna(inplace=True)

print(panthdata.isna().sum()) # check for missing values

In [ ]:
# we are now ready to fit models
panthdata.head()

## Regression model 3: Log of length of longevity and log of gestation

In [ ]:
reg_mod2 = smf.ols("gestation_log ~ longevity_log", data=panthdata) # setup the model
reg_mod2_fit = reg_mod2.fit() # estimate/fit the model 
reg_mod2_summ = reg_mod2_fit.summary()
reg_mod2_summ.tables[1]

### Interpreting our model

- The regression equation is: $\texttt{log.gestation} =   0.7819\times \texttt{log.longevity} + 0.5374$
- The **slope** indicates that for a one log-month **increase** in longevity, we expect the gestational period to increase by 0.7819 log-days
- The **y-intercept** indicates that when longevity is zero log-months, gestational period is 0.5374 log-days


### Interpreting our model


Let's take a look at the model's accuracy, by checking it's R squared and residuals.

In [ ]:
sns.regplot(x="longevity_log", y="gestation_log", data=panthdata, ci=None, scatter_kws={"alpha":0.1}) 
plt.text(0.05, 0.9, f"R^2: {reg_mod2_fit.rsquared:.3f}", transform=plt.gca().transAxes)
plt.show()

In [ ]:
plt.scatter(x=reg_mod2_fit.fittedvalues , y=reg_mod2_fit.resid, alpha=.3)
plt.axhline(y=0, color="red", linestyle="dotted")
plt.xlabel("Fitted values")
plt.ylabel("Residuals");

## Regression Model 4: Log of body mass and log of gestation

In [ ]:
## Recall the scatterplot
plt.scatter(y=panthdata["gestation_log"], x=panthdata["bodymass_log"], alpha=.2)
plt.xlabel("Body mass (log g)")
plt.ylabel("Gestation (log days)");

In [ ]:
reg_mod4 = smf.ols("gestation_log~bodymass_log", data=panthdata) # setup the model
reg_mod4_fit = reg_mod4.fit() # estimate/fit the model 
reg_mod4_summ = reg_mod4_fit.summary()
reg_mod4_summ.tables[1]

### Interpreting our model

- The regression equation is: $\texttt{gestation} = 2.7866 + 0.2119 \times \log(\texttt{body.mass})$
- The p-value for the slope is still significant after transformation. So log body mass has a non-zero slope when predicting gestation.

In [ ]:
sns.regplot(x="bodymass_log", y="gestation_log", data=panthdata, ci=None, scatter_kws={"alpha":0.1}) 
plt.text(0.05, 0.9, f"R^2: {reg_mod4_fit.rsquared:.3f}", transform=plt.gca().transAxes)
plt.show()

In [ ]:
plt.scatter(x=reg_mod4_fit.fittedvalues , y=reg_mod4_fit.resid, alpha=.3)
plt.axhline(y=0, color="red", linestyle="dotted")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")

## Regression Model 5: Log of body mass and log of longevity

In [ ]:
## Recall the scatterplot
plt.scatter(y=panthdata["longevity_log"], x=panthdata["bodymass_log"], alpha=.2)
plt.xlabel("Body mass (log g)")
plt.ylabel("Longevity (log months)");

In [ ]:
reg_mod5 = smf.ols("longevity_log~bodymass_log", data=panthdata) # setup the model
reg_mod5_fit = reg_mod5.fit() # estimate/fit the model 
reg_mod5_summ = reg_mod5_fit.summary()
reg_mod5_summ.tables[1]

### Interpreting our model

- The regression equation is: $\texttt{longevity} = 2.7866 + 0.2119 \times \log(\texttt{body.mass})$
- The p-value for the slope is still significant after transformation. So log body mass has a non-zero slope when predicting gestation.

In [ ]:
sns.regplot(x="bodymass_log", y="longevity_log", data=panthdata, ci=None, scatter_kws={"alpha":0.1}) 
plt.text(0.05, 0.9, f"R^2: {reg_mod5_fit.rsquared:.3f}", transform=plt.gca().transAxes)
plt.show()

In [ ]:
plt.scatter(x=reg_mod4_fit.fittedvalues , y=reg_mod4_fit.resid, alpha=.3)
plt.axhline(y=0, color="red", linestyle="dotted")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")

## Regression Model 6: Multiple regression with log-transformed variables

In [ ]:
reg_mod6 = smf.ols("gestation_log~bodymass_log+longevity_log", data=panthdata) # setup the model
reg_mod6_fit = reg_mod6.fit()
reg_mod6_summ = reg_mod6_fit.summary()
reg_mod6_summ.tables[1]

### Interpreting our model

- The regression equation is: $\texttt{log(gestation)} =  \texttt{0.4311} \times \texttt{log(longevity)} + \texttt{0.1252} \times \log(\texttt{body.mass}) + 1.3100$
- The **slope** of longevity indicates that for every 1 log month **increase** in longevity, _while keeping body mass constant_, we expect the gestational period to increase by 0.4311 log days
- The **slope** of log body mass indicates that for every 1 unit **increase** in log body mass, _while keeping longevity length constant_, we expect the gestational period to increase by 0.1252 log days
- The **y-intercept** indicates that when longevity is 0 months and log body mass is 0, gestational period is 1.31 log days.

In [ ]:
import statsmodels.api as sm

fig = sm.graphics.plot_partregress("gestation_log", "bodymass_log", ["longevity_log"], data=panthdata, obs_labels=False)

In [ ]:
fig = sm.graphics.plot_partregress("gestation_log", "longevity_log", ["bodymass_log"], data=panthdata, obs_labels=False)

### Model 6 R-squared

In [ ]:
reg_mod6_fit.rsquared

In [ ]:
plt.scatter(x=reg_mod6_fit.fittedvalues , y=reg_mod6_fit.resid, alpha=.3)
plt.axhline(y=0, color="red", linestyle="dotted")
plt.xlabel("Fitted values")
plt.ylabel("Residuals");

### Data question: do large mammals spend longer in utero than small mammals?

- Let's use one of our models to explore a simple question 

### Data question: do large mammals spend longer in utero than small mammals?
- Let's use one of our models to explore a simple question 

- We have a good sense that body mass and gestation time are related
  - Organisms that are larger also tend to spend a longer time in utero
  - Many timing variables vary according to body mass-- larger bodies mean more time is needed to develop/grow
  

In [ ]:
gest = panthdata[['order','gestation_log']]
gest_ord = gest.groupby('order')
gest_ord_means = np.exp(gest_ord.median()).sort_values(by="gestation_log",ascending=False)
gest_ord_means.iloc[0:15].plot.bar(rot=70)
plt.ylabel("Median gestation time (days)")
plt.show()

### Data question: do large mammals spend longer in utero than small mammals?

- Let's re-fit our linear model of gestation time on body size:

In [ ]:
reg_mod4 = smf.ols("gestation_log~bodymass_log", data=panthdata) # setup the model
reg_mod4_fit = reg_mod4.fit() # estimate/fit the model 
reg_mod4_summ = reg_mod4_fit.summary()
reg_mod4_summ.tables[1]

In [ ]:
sns.regplot(x="bodymass_log", y="gestation_log", data=panthdata, ci=None, scatter_kws={"alpha":0.1}) 
plt.text(0.05, 0.9, f"R^2: {reg_mod4_fit.rsquared:.3f}", transform=plt.gca().transAxes)
plt.show()

### Data question: do large mammals spend longer in utero than small mammals?

- And now let's interpret the residuals as "relative" gestation times

In [ ]:
rel_gest = reg_mod4_fit.resid
rel_gest = pd.DataFrame({'order':gest['order'],'rel_gest':rel_gest})
rel_gest_ord = rel_gest.groupby('order')
rel_gest_ord_means = rel_gest_ord.median().sort_values(by="rel_gest",ascending=False)

rel_gest_ord_means.iloc[0:15].plot.bar(rot=70)
plt.ylabel("Median gestation time above expected (log-days)")
plt.show()

### Causality

Recall that **correlation does not imply causation**. All data we analyzed are observational, and has hidden/confounding variables could drive the relationship 

## Multidimensionality


### Multidimensionality

  - We may want to understand the 'correlation structure' between several variables
  - We could do multiple regression, but that comes with caveats
    - Also hard to visualize -- how do we plot 6 dimensions?


### Principal Components Analysis

- PCA is a common way to deal with multidimensionality
- Take a dataset containing many dimensions and reduce them to only a few 


### PCA
  
  - there is some "redundancy" in correlated measurements
  - PCA uses matrix math (which we will not touch) to identify this redundancy


### PCA

  - _input_: a set of **X** measurements
  - _output_: a set of **X** transformed measurements that are uncorrelated
  - each new measurement captures information from several of the original measurements
  - we call these new measurements **principal components**, or **PCs**



In [ ]:
important_columns = ["MSW05_Order","MSW05_Binomial","5-1_AdultBodyMass_g","23-1_SexualMaturityAge_d",'17-1_MaxLongevity_m','25-1_WeaningAge_d','9-1_GestationLen_d']

sub_pantheria = pantheria[important_columns]
columnnames = {'MSW05_Order': 'order',
               'MSW05_Binomial': 'genus_species',
               '5-1_AdultBodyMass_g': 'bodymass',
               '23-1_SexualMaturityAge_d': 'maturity',
               '17-1_MaxLongevity_m': 'longevity',
               '25-1_WeaningAge_d':"weaning",
               '9-1_GestationLen_d':"gestation"}

rn_pantheria = sub_pantheria.rename(columns=columnnames)

In [ ]:
log_pantheria = rn_pantheria[['order', 'genus_species']].assign(
    bodymass_log = np.log(rn_pantheria['bodymass']),
    longevity_log = np.log(rn_pantheria['longevity']),
    gestation_log = np.log(rn_pantheria['gestation']),
    weaning_log = np.log(rn_pantheria['weaning']),
    maturity_log = np.log(rn_pantheria['maturity'])
)
log_pantheria.dropna(inplace=True)

log_pantheria.head()

## How do the data look? 
  - Can we can easily spot differences between species?
  - This is **exploratory data analysis**

In [ ]:
panth_sub = log_pantheria.loc[(log_pantheria['order']=="Carnivora") | (log_pantheria['order']=="Primates") | (log_pantheria['order']=="Cetacea") | (log_pantheria['order']=="Rodentia")]
for name, group in panth_sub.groupby('order'):
    plt.plot(group.bodymass_log, group.gestation_log, marker='o', linestyle='', label=name)
plt.legend()
plt.xlabel("bodymass_log")
plt.ylabel("gestation_log")
plt.show()

## Dimension reduction

- Working through each pair of variables like this can quickly become overwhelming
- Many variables may be **redundant** because they are related to one another
  + e.g., if we know a species' body mass, we can make a good guess about its gestation time

## Principal components analysis (PCA)

- PCA draws lines like this between all of our measurements
- Each line thus contains a different 'combination' of the original measurements
  + We call each of these lines a "principal component"

In [ ]:
# we need to extract just the measurements -- do away temporarily with the species and order labels

traits=log_pantheria.loc[:,log_pantheria.columns[2:]].values 
traits=pd.DataFrame(StandardScaler().fit_transform(traits))

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca=PCA(n_components=4) # set up our PCA 
fit=pca.fit_transform(traits) # fit the PCA to our scaled measurements
scores = pd.DataFrame(data = fit, columns = ['pc1', 'pc2', 'pc3', 'pc4'])
scores

### Variance

- We can find out exactly how much of this spread is captured by each principal component (PC)
- PCs with higher values explain more information from the original dataset

In [ ]:
prop_var=pca.explained_variance_ratio_ * 100
prop_var

### Relationships between variables

- PCA provides a way of examining all relationships between variables
  - same direction: positively correlated (as x gets bigger, y gets bigger)
  - opposite direction: negatively correlated (as x gets bigger, y gets smaller)
  - perpendicular: uncorrelated (x and y have no relationship)

In [ ]:
ax=plt.plot(scores.pc1, scores.pc2, marker='o', linestyle='')
coeff = np.transpose(pca.components_[0:3,:])
n = coeff.shape[0]
for i in range(n):
    plt.arrow(0, 0, coeff[i,0]*60, coeff[i,1]*60,color = 'r',alpha = 0.9)
    plt.text(coeff[i,0]* 60, coeff[i,1] * 60, list(log_pantheria.columns[2:])[i], color = 'g', ha = 'center', va = 'center')
plt.xlabel("PC1 ("+str(round(prop_var[0]))+"%)")
plt.ylabel("PC2 ("+str(round(prop_var[1]))+"%)")
plt.show()

In [ ]:
scores['order'] = list(log_pantheria['order'])
scores.head()

### Visualizing differences between groups

- One handy thing about PCA is that it allows us to visualize how simlar different groups are within our data

In [ ]:
sub_scores = scores.loc[(scores['order']=="Carnivora") | (scores['order']=="Primates") | (scores['order']=="Cetacea") | (scores['order']=="Rodentia")]
for name, group in sub_scores.groupby('order'):
    plt.plot(group.pc1, group.pc2, marker='o', linestyle='', label=name)
plt.legend()
plt.xlabel("PC1 ("+str(round(prop_var[0]))+"%)")
plt.ylabel("PC2 ("+str(round(prop_var[1]))+"%)")